# 셋팅 #

필요한 설정을 위해 이 셀을 실행하세요!


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')
ryanholbrook_ts_course_data_path = kagglehub.dataset_download('ryanholbrook/ts-course-data')

print('Data source import complete.')


In [ ]:
import os
from pathlib import Path

# Define the target directory where learntools expects to find the data
input_base_path = Path('/input')
input_base_path.mkdir(parents=True, exist_ok=True)

# Create a symbolic link for the store-sales-time-series-forecasting competition data
symlink_target_comp = input_base_path / 'store-sales-time-series-forecasting'
symlink_source_comp = store_sales_time_series_forecasting_path

if symlink_target_comp.exists() or symlink_target_comp.is_symlink():
    if symlink_target_comp.is_symlink():
        os.unlink(symlink_target_comp)
    else:
        # If it's a directory, remove it to replace with symlink
        import shutil
        shutil.rmtree(symlink_target_comp)

os.symlink(symlink_source_comp, symlink_target_comp)
print(f"Created symlink: {symlink_target_comp} -> {symlink_source_comp}")

# Create a symbolic link for the ryanholbrook/ts-course-data dataset
symlink_target_data = input_base_path / 'ts-course-data'
symlink_source_data = ryanholbrook_ts_course_data_path

if symlink_target_data.exists() or symlink_target_data.is_symlink():
    if symlink_target_data.is_symlink():
        os.unlink(symlink_target_data)
    else:
        import shutil
        shutil.rmtree(symlink_target_data)

os.symlink(symlink_source_data, symlink_target_data)
print(f"Created symlink: {symlink_target_data} -> {symlink_source_data}")


In [ ]:
!git clone https://github.com/Kaggle/learntools.git
!mv learntools learntools_dir
!mv learntools_dir/learntools learntools

# 시계열 예측 소개 #

2, 3강에서는 시간 인덱스 하나에서 파생한 피처만 사용해 예측 문제를 단순 회귀 문제처럼 다뤘습니다. 원했던 추세와 계절 피처만 만들어 주면 임의의 미래 시점도 쉽게 예측할 수 있었죠.

하지만 4강에서 래그 피처(시계열 자체를 피처로 쓰는 방법)를 도입하면서 상황이 달라졌습니다. 래그 피처를 사용하려면 예측하려는 시점에서 래그된 타깃 값을 미리 알고 있어야 합니다. 예를 들어 래그 1 피처는 시계를 한 단계 뒤로 미루므로 한 단계 앞까지는 예측할 수 있지만 두 단계 앞은 불가능합니다.

4강에서는 필요한 래그가 항상 준비돼 있다고 가정했습니다(한 번에 한 스텝만 예측했다고 볼 수 있습니다). 현실 세계의 예측 문제는 이보다 복잡하므로 이번 강의에서는 다양한 상황에서 예측을 만드는 방법을 배웁니다.

# 예측 작업 정의하기 #

모델을 설계하기 전에 두 가지를 먼저 정해야 합니다.
- 예측 시점에서 사용할 수 있는 정보(피처)는 무엇인가?
- 어떤 기간에 대해 예측값(타깃)이 필요한가?

**예측 시점(forecast origin)** 은 예측을 내리는 시각입니다. 실무에서는 예측하려는 시계열에 대해 학습 데이터가 마지막으로 존재하는 시점을 의미합니다. 시점 이전까지의 모든 정보는 피처로 사용할 수 있습니다.

**예측 구간(forecast horizon)** 은 예측이 필요한 기간입니다. 예를 들어 "1단계 예측", "5단계 예측"이라고 부를 때의 단계 수가 곧 예측 구간입니다. 예측 구간이 타깃을 정의합니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/xwEgcOk.png" width=500, alt="">
<figcaption style="textalign: center; font-style: italic"><center>네 개의 래그 피처로 두 단계 앞에서 시작해 세 단계를 예측하는 경우. 그림 하나가 예측 한 건(데이터프레임 한 행)에 해당합니다.
</center></figcaption>
</figure>

시점과 구간 사이의 간격은 예측의 **리드 타임(lead time)** 또는 지연(latency)입니다. 리드 타임은 시점에서 구간까지 몇 단계 떨어져 있는지로 표현합니다(예: "1단계 앞 예측", "3단계 앞 예측"). 데이터 수집·처리에 시간이 걸릴 경우 시점보다 여러 단계 앞에서 예측을 시작해야 할 수도 있습니다.

# 예측을 위한 데이터 준비 #

ML 알고리즘으로 시계열을 예측하려면(추세·계절 같은 결정적 피처만 사용할 때를 제외하면) 시계열을 이 알고리즘이 다룰 수 있는 데이터프레임 형태로 변환해야 합니다.

4강에서 래그를 만들어 피처로 썼던 것이 변환 과정의 전반부였습니다. 후반부는 타깃을 준비하는 일입니다. 예측 작업에 따라 타깃을 준비하는 방식이 달라집니다.

데이터프레임의 각 행은 예측 한 건을 나타냅니다. 행의 시간 인덱스는 예측 구간의 시작 시각이지만, 구간 전체 값을 같은 행에 배치합니다. 여러 스텝을 한 번에 예측할 때는 모델이 단계 수만큼 출력을 내도록 요구하는 셈입니다.


In [ ]:
import numpy as np
import pandas as pd

N = 20
ts = pd.Series(
    np.arange(N),
    index=pd.period_range(start='2010', freq='A', periods=N, name='Year'),
    dtype=pd.Int8Dtype,
)

# 래그 피처
X = pd.DataFrame({
    'y_lag_2': ts.shift(2),
    'y_lag_3': ts.shift(3),
    'y_lag_4': ts.shift(4),
    'y_lag_5': ts.shift(5),
    'y_lag_6': ts.shift(6),
})

# 다단계 타깃
y = pd.DataFrame({
    'y_step_3': ts.shift(-2),
    'y_step_2': ts.shift(-1),
    'y_step_1': ts,
})

data = pd.concat({'Targets': y, 'Features': X}, axis=1)

data.head(10).style.set_properties(['Targets'], **{'background-color': 'LavenderBlush'})                    .set_properties(['Features'], **{'background-color': 'Lavender'})


위 그림은 앞의 예시(3단계 예측, 2단계 리드 타임, 5개 래그 피처)가 데이터프레임으로 구성된 모습입니다. 원래 시계열은 `y_step_1`이고, 생성된 결측은 채우거나 삭제할 수 있습니다.

# 다단계 예측 전략 #

예측에 필요한 여러 단계의 타깃을 만드는 방식은 다양합니다. 여기서는 장단점을 가진 네 가지 대표 전략을 소개합니다.

### 다중 출력(Multioutput) 모델

한 모델이 여러 출력을 자연스럽게 생성하는 방식입니다. 선형 회귀나 신경망은 다중 출력을 지원합니다. 구현이 간단하고 효율적이지만, 모든 알고리즘이 다중 출력을 지원하는 것은 아닙니다. 예컨대 XGBoost는 이 방식을 직접 사용할 수 없습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/uFsHiqr.png" width=300, alt="">
<figcaption style="textalign: center; font-style: italic"><center></center></figcaption>
</figure>

### 직접(Direct) 전략

예측 구간의 각 단계마다 별도의 모델을 학습합니다. 1단계 예측 모델, 2단계 예측 모델...처럼요. 1단계 앞을 예측하는 문제와 2단계 앞을 예측하는 문제는 성격이 다르므로 별도의 모델을 쓰는 것이 도움이 되기도 합니다. 단점은 단계 수만큼 모델을 학습해야 하므로 계산 비용이 커진다는 점입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/HkolNMV.png" width=900, alt="">
<figcaption style="textalign: center; font-style: italic"><center></center></figcaption>
</figure>

### 재귀(Recursive) 전략

한 단계 예측 모델만 학습한 뒤, 그 예측을 다음 단계의 래그 피처로 사용합니다. 즉, 1단계 예측을 다시 모델에 넣어 그다음 단계를 예측합니다. 모델 하나만 학습하면 되지만, 각 단계의 오차가 다음 단계로 누적되기 때문에 구간이 길수록 정확도가 떨어질 수 있습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/sqkSFDn.png" width=300, alt="">
<figcaption style="textalign: center; font-style: italic"><center></center></figcaption>
</figure>

### DirRec 전략

Direct와 Recursive를 결합한 방식입니다. 각 단계마다 모델을 학습하되, 이전 단계의 예측을 새로운 래그 피처로 사용합니다. 단계가 늘어날수록 모델마다 래그 입력이 하나씩 늘어납니다. 항상 최신 래그를 사용하므로 DirRec 전략은 Direct 전략보다 계열 의존성을 더 잘 포착할 수 있지만, Recursive 전략처럼 오차 전파를 겪을 수도 있습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/B7KAvAO.png" width=900, alt="">
<figcaption style="textalign: center; font-style: italic"><center></center></figcaption>
</figure>

# 예제 - Flu Trends #

이번 예제에서는 4강의 *Flu Trends* 데이터를 다시 사용하되, 진짜 다단계 예측을 만들어 보겠습니다. 예측 구간은 8주, 리드 타임은 1주로 설정합니다. 즉, 다음 주부터 8주 동안의 독감 환자 수를 예측하겠다는 뜻입니다.

아래 숨겨진 셀에서 예제를 위한 준비와 `plot_multistep` 헬퍼 함수를 정의했습니다.


In [ ]:
from pathlib import Path
from warnings import simplefilter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

simplefilter("ignore")

# Matplotlib 기본 설정
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=16,
    titlepad=10,
)
plot_params = dict(
    color="0.75",
    style=".-",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
)
%config InlineBackend.figure_format = 'retina'

def plot_multistep(y, every=1, ax=None, palette_kwargs=None):
    palette_kwargs_ = dict(palette='husl', n_colors=16, desat=None)
    if palette_kwargs is not None:
        palette_kwargs_.update(palette_kwargs)
    palette = sns.color_palette(**palette_kwargs_)
    if ax is None:
        fig, ax = plt.subplots()
    ax.set_prop_cycle(plt.cycler('color', palette))
    for date, preds in y[::every].iterrows():
        preds.index = pd.period_range(start=date, periods=len(preds))
        preds.plot(ax=ax)
    return ax

data_dir = Path("../input/ts-course-data")
flu_trends = pd.read_csv(data_dir / "flu-trends.csv")
flu_trends.set_index(
    pd.PeriodIndex(flu_trends.Week, freq="W"),
    inplace=True,
)
flu_trends.drop("Week", axis=1, inplace=True)


먼저 타깃 시리즈(독감으로 병원을 찾은 주간 방문자 수)를 다단계 예측에 맞게 준비합니다. 한 번 준비해 두면 이후 학습과 예측은 아주 간단해집니다.


In [ ]:
def make_lags(ts, lags, lead_time=1):
    return pd.concat(
        {
            f'y_lag_{i}': ts.shift(i)
            for i in range(lead_time, lags + lead_time)
        },
        axis=1)


# 4주치 래그 피처
y = flu_trends.FluVisits.copy()
X = make_lags(y, lags=4).fillna(0.0)


def make_multistep_target(ts, steps):
    return pd.concat(
        {f'y_step_{i + 1}': ts.shift(-i)
         for i in range(steps)},
        axis=1)


# 8주 예측
y = make_multistep_target(y, steps=8).dropna()

# 시프트 때문에 인덱스가 어긋났으니 타깃과 피처가 모두 존재하는 시점만 남깁니다.
y, X = y.align(X, join='inner', axis=0)


### 다중 출력 모델

다중 출력 전략으로는 선형 회귀를 사용하겠습니다. 다중 출력용 데이터만 준비되어 있으면 학습과 예측은 언제나와 똑같습니다.


In [ ]:

# Create splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

y_fit = pd.DataFrame(model.predict(X_train), index=X_train.index, columns=y.columns)
y_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)

다단계 모델은 입력으로 사용한 각 시점마다 예측 전체를 생성한다는 사실을 기억하세요. 학습 세트에 269주, 테스트 세트에 90주가 있으므로 이제 각 주에 대해 8단계 예측이 하나씩 존재합니다.


In [ ]:
import numpy as np

train_rmse = np.sqrt(mean_squared_error(y_train, y_fit))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print((f"Train RMSE: {train_rmse:.2f}\n" f"Test RMSE: {test_rmse:.2f}"))

palette = dict(palette='husl', n_colors=64)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6))
ax1 = flu_trends.FluVisits[y_fit.index].plot(**plot_params, ax=ax1)
ax1 = plot_multistep(y_fit, ax=ax1, palette_kwargs=palette)
_ = ax1.legend(['FluVisits (train)', 'Forecast'])
ax2 = flu_trends.FluVisits[y_pred.index].plot(**plot_params, ax=ax2)
ax2 = plot_multistep(y_pred, ax=ax2, palette_kwargs=palette)
_ = ax2.legend(['FluVisits (test)', 'Forecast'])

### Direct 전략

XGBoost는 회귀에서 다중 출력을 제공하지 않지만, Direct 전략을 적용하면 다단계 예측을 만들 수 있습니다. scikit-learn의 `MultiOutputRegressor` 래퍼로 간단히 해결할 수 있습니다.


In [ ]:
from sklearn.multioutput import MultiOutputRegressor

model = MultiOutputRegressor(XGBRegressor())
model.fit(X_train, y_train)

y_fit = pd.DataFrame(model.predict(X_train), index=X_train.index, columns=y.columns)
y_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)

이번 예제에서 XGBoost는 학습 세트에 다소 과적합했지만, 테스트 세트에서는 선형 회귀보다 독감 시즌의 변화를 더 잘 포착하는 모습입니다. 하이퍼파라미터를 튜닝하면 성능을 더 높일 수 있을 것입니다.


In [ ]:
import numpy as np

train_rmse = np.sqrt(mean_squared_error(y_train, y_fit))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print((f"Train RMSE: {train_rmse:.2f}\n" f"Test RMSE: {test_rmse:.2f}"))

palette = dict(palette='husl', n_colors=64)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6))
ax1 = flu_trends.FluVisits[y_fit.index].plot(**plot_params, ax=ax1)
ax1 = plot_multistep(y_fit, ax=ax1, palette_kwargs=palette)
_ = ax1.legend(['FluVisits (train)', 'Forecast'])
ax2 = flu_trends.FluVisits[y_pred.index].plot(**plot_params, ax=ax2)
ax2 = plot_multistep(y_pred, ax=ax2, palette_kwargs=palette)
_ = ax2.legend(['FluVisits (test)', 'Forecast'])

# 소개 #

환경을 모두 준비하려면 이 셀을 실행하세요!


In [ ]:
# 피드백 시스템 설정
from learntools.core import binder
binder.bind(globals())
from learntools.time_series.ex6 import *

# 노트북 환경 설정
from pathlib import Path
import ipywidgets as widgets
from learntools.time_series.style import *  # 플롯 스타일 설정
from learntools.time_series.utils import (create_multistep_example,
                                          load_multistep_data,
                                          make_lags,
                                          make_multistep_target,
                                          plot_multistep)

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import RegressorChain
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor


comp_dir = Path('../input/store-sales-time-series-forecasting')

store_sales = pd.read_csv(
    comp_dir / 'train.csv',
    usecols=['store_nbr', 'family', 'date', 'sales', 'onpromotion'],
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'sales': 'float32',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
store_sales['date'] = store_sales.date.dt.to_period('D')
store_sales = store_sales.set_index(['store_nbr', 'family', 'date']).sort_index()

family_sales = (
    store_sales
    .groupby(['family', 'date'])
    .mean()
    .unstack('family')
    .loc['2017']
)

test = pd.read_csv(
    comp_dir / 'test.csv',
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
test['date'] = test.date.dt.to_period('D')
test = test.set_index(['store_nbr', 'family', 'date']).sort_index()


-------------------------------------------------------------------------------

다음과 같은 세 가지 예측 작업을 생각해 봅시다.

a. 래그 4개, 리드 타임 2단계, 예측 구간 3단계
b. 래그 3개, 리드 타임 1단계, 예측 구간 1단계
c. 래그 4개, 리드 타임 1단계, 예측 구간 3단계

다음 셀에는 위 작업을 각각 표현한 세 개의 데이터셋이 있습니다.


In [ ]:
datasets = load_multistep_data()

data_tabs = widgets.Tab([widgets.Output() for _ in enumerate(datasets)])
for i, df in enumerate(datasets):
    data_tabs.set_title(i, f'Dataset {i+1}')
    with data_tabs.children[i]:
        display(df)

display(data_tabs)

# 1) 설명과 데이터셋 연결하기

각 작업이 어떤 데이터셋에 해당하는지 맞혀 보세요.


In [ ]:
# 여기에 코드를 작성하세요: 작업을 데이터셋에 연결하세요. 답은 1, 2, 3 중에서 고르세요.
task_a = ____
task_b = ____
task_c = ____



In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_1.hint()
#q_1.solution()


-------------------------------------------------------------------------------

학습·테스트 세트의 시간 인덱스를 살펴보세요. 이 정보만으로 *Store Sales* 예측 작업을 파악할 수 있나요?


In [ ]:
print("Training Data", "\n" + "-" * 13 + "\n", store_sales)
print("\n")
print("Test Data", "\n" + "-" * 9 + "\n", test)

# 2) *Store Sales* 대회의 예측 작업 파악하기

예측 시점과 예측 구간을 찾아보세요. 구간 안에는 몇 단계가 들어가나요? 리드 타임은 얼마인가요? 답을 생각한 뒤 이 셀을 실행하세요.


In [ ]:
# 해설 보기(이 셀을 실행해야 점수를 받을 수 있습니다!)
q_2.check()


-------------------------------------------------------------------------------

튜토리얼에서는 단일 시계열에 대한 다단계 데이터셋을 만드는 방법을 살펴봤습니다. 다행히 여러 시계열이 있어도 절차는 동일합니다.

# 3) *Store Sales*용 다단계 데이터셋 만들기

*Store Sales* 예측 작업에 맞는 타깃을 만드세요. 래그는 4일치 사용하고, 타깃과 피처에서 결측은 모두 제거하세요.


In [ ]:
# 직접 작성하세요
y = family_sales.loc[:, 'sales']

# 직접 작성하세요: 래그 4개 만들기
X = ____

# 직접 작성하세요: 다단계 타깃 만들기
y = ____

y, X = y.align(X, join='inner', axis=0)



In [ ]:
y = family_sales.loc[:, 'sales']

X = make_lags(y, lags=4).dropna()

y = make_multistep_target(y, steps=16).dropna()

y, X = y.align(X, join='inner', axis=0)

In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_3.hint()
#q_3.solution()


-------------------------------------------------------------------------------

튜토리얼에서는 *Flu Trends* 시리즈에 MultiOutput과 Direct 전략을 적용했습니다. 이번에는 DirRec 전략을 *Store Sales*의 여러 시계열에 적용해 보겠습니다.

직전 문제를 완료한 뒤 다음 셀을 실행해 XGBoost용 데이터를 준비하세요.


In [ ]:
le = LabelEncoder()
X = (X
    .stack('family')  # 와이드 -> 롱
    .reset_index('family')  # 인덱스를 열로 변환
    .assign(family=lambda x: le.fit_transform(x.family))  # 라벨 인코딩
)
y = y.stack('family')  # 와이드 -> 롱

display(y)


# 4) DirRec 전략으로 예측하기

DirRec 전략을 XGBoost에 적용하는 모델을 초기화하세요.


In [ ]:
from sklearn.multioutput import RegressorChain

# 직접 작성하세요
model = ____


In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_4.hint()
#q_4.solution()


이 모델을 훈련하려면 이 셀을 실행하세요.

In [ ]:
model.fit(X, y)

y_pred = pd.DataFrame(
    model.predict(X),
    index=y.index,
    columns=y.columns,
).clip(0.0)


이 코드를 실행하면 학습 데이터에서 16단계 예측 샘플을 확인할 수 있습니다.


In [ ]:
FAMILY = 'BEAUTY'
START = '2017-04-01'
EVERY = 16

y_pred_ = y_pred.xs(FAMILY, level='family', axis=0).loc[START:]
y_ = family_sales.loc[START:, 'sales'].loc[:, FAMILY]

fig, ax = plt.subplots(1, 1, figsize=(11, 4))
ax = y_.plot(**plot_params, ax=ax, alpha=0.5)
ax = plot_multistep(y_pred_, ax=ax, every=EVERY)
_ = ax.legend([FAMILY, FAMILY + ' Forecast'])


# **튜토리얼 끝~**

---

